In [1]:
import numpy as np
import matplotlib as mplt
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import random
import math
import copy
from typing import Annotated, Any, Callable
from pydantic import BaseModel, Field, WithJsonSchema
import pydantic

In [ ]:
import nltk
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.tokenize import word_tokenize

SEED=41
nltk.download('punkt_tab')
 


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jsima\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [38]:
corpus = ["this is a test", "this is another test"]
corpus_ = [word_tokenize(s) for s in corpus]

n = 3
train_data, vocab = padded_everygram_pipeline(n, corpus_)

for kv in vocab:
    print('->', kv)
    
model = MLE(n)
model.fit(train_data, vocab)

# Generate sequence
for i in range(0, 10):
    ss = model.generate(10, random_seed=SEED)
    print(ss)


-> <s>
-> <s>
-> this
-> is
-> a
-> test
-> </s>
-> </s>
-> <s>
-> <s>
-> this
-> is
-> another
-> test
-> </s>
-> </s>
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>

In [33]:
from collections import Counter, defaultdict
import random

counts = defaultdict(Counter)


def train_ngram(corpus, N):
    """
    Train an N-gram language model.

    Parameters
    ----------
    corpus : list[list[str]]
        Tokenized sentences.
    N : int
        Order of the n-gram (N=2 => bigram, N=3 => trigram, etc.)

    Returns
    -------
    counts : dict[tuple, Counter]
        Mapping from (N-1)-word context to counts of next words.
    """
    assert N >= 1, "N must be >= 1"

    counts = defaultdict(Counter)

    for sentence in corpus:
        # Pad with start and end symbols
        tokens = ["<s>"] * (N - 1) + sentence + ["</s>"]

        for i in range(len(tokens) - N + 1):
            context = tuple(tokens[i : i + N - 1])  # empty tuple if N=1
            target  = tokens[i + N - 1]

            print(context, target)
            counts[context][target] += 1

    return counts


def sample_next_ngram(context, counts):
    counter = counts.get(context)
    if not counter:
        return "</s>"

    words, freqs = zip(*counter.items())
    total = sum(freqs)
    probs = [f / total for f in freqs]
    print('->', words, freqs)
    return random.choices(words, probs)[0]


def generate_sentence(counts, N, L, start="<s>"):
    sentence = []
    context = start
    
    context = ("<s>",) * (N - 1)

    for _ in range(L):
        next_word = sample_next_ngram(context, counts)

        print('!', next_word)
        if next_word == "</s>" and len(sentence)>5:
            break

        sentence.append(next_word)
        context = next_word

    return sentence

corpus = ["this is a test", "this is another test"]
corpus = ["A eats B", "B runs from C"]

corpus_ = [word_tokenize(s) for s in corpus]
print(corpus_)

counts = train_ngram(corpus_, N=2)
print(counts)
s = generate_sentence(counts, N=2, L=10, start="<s>")
print(s)


[['A', 'eats', 'B'], ['B', 'runs', 'from', 'C']]
('<s>',) A
('A',) eats
('eats',) B
('B',) </s>
('<s>',) B
('B',) runs
('runs',) from
('from',) C
('C',) </s>
defaultdict(<class 'collections.Counter'>, {('<s>',): Counter({'A': 1, 'B': 1}), ('A',): Counter({'eats': 1}), ('eats',): Counter({'B': 1}), ('B',): Counter({'</s>': 1, 'runs': 1}), ('runs',): Counter({'from': 1}), ('from',): Counter({'C': 1}), ('C',): Counter({'</s>': 1})})
-> ('A', 'B') (1, 1)
! A
! </s>
! </s>
! </s>
! </s>
! </s>
! </s>
['A', '</s>', '</s>', '</s>', '</s>', '</s>']


In [34]:
import markovify

text = "this is a test this is another test"
model = markovify.Text(text, state_size=2)

model.make_sentence()


'this is a test this is a test this is another test'

In [42]:
import sklearn
from sklearn.datasets import fetch_20newsgroups

categories = ['alt.atheism', 'soc.religion.christian',
              'comp.graphics', 'sci.med']

twenty_train = fetch_20newsgroups(subset='train',
    categories=categories, shuffle=True, random_state=42)

KeyboardInterrupt: 

In [47]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(corpus) #twenty_train.data
X_train_counts.shape
print(X_train_counts)
print(count_vect.vocabulary_)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7 stored elements and shape (2, 4)>
  Coords	Values
  (0, 3)	1
  (0, 1)	1
  (0, 2)	1
  (1, 3)	1
  (1, 1)	1
  (1, 2)	1
  (1, 0)	1
{'this': 3, 'is': 1, 'test': 2, 'another': 0}
